# 🎙️ REACTOR: Full-Duplex Voice Agent Evaluation on Kaggle
**PRISM GenAI Hackathon (Theme 05: Full-Duplex Voice Agents)**  
Evaluation of **REACTOR** against the official **NTU Full-Duplex-Bench v3 (FDB-v3)** benchmark (100 real human audio recordings).

---

### ⚙️ Hardware & Environment Requirements
> [!IMPORTANT]
> - **Accelerator:** Select **GPU T4 x2** in the right-hand panel (`Notebook Options` → `Accelerator` → `GPU T4 x2`).
>   * *Why not TPU v5e-8?* Full-Duplex-Bench uses NVIDIA NeMo Parakeet ASR (`nvidia/parakeet-tdt-0.6b-v2`), which compiles CUDA C++ kernels and Triton routines. Google TPU cannot execute PyTorch CUDA binaries.
> - **Internet:** Must be toggled **ON** (`Settings` → `Internet` → `On`).
> - **Kaggle Secrets:** Add the following keys in Kaggle menu **Add-ons → Secrets**:
>   - `LIVEKIT_URL`: Your LiveKit Cloud WebSocket URL (`wss://<project>.livekit.cloud`)
>   - `LIVEKIT_API_KEY`: Your LiveKit API Key
>   - `LIVEKIT_API_SECRET`: Your LiveKit API Secret
>   - `GOOGLE_API_KEY`: Your Google Gemini API Key
>   - *(Optional)* `OPENAI_API_KEY`: Only if running with `--use-llm` for the semantic LLM judge.


In [ ]:
# 1. Verify NVIDIA GPU & CUDA environment
import os
import subprocess
import sys
import torch

print(f"Python executable: {sys.executable}")
print(f"Python version: {sys.version.split()[0]}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if not torch.cuda.is_available():
    raise RuntimeError(
        "❌ CUDA GPU is not available! In the right sidebar, change Accelerator to 'GPU T4 x2'."
    )

print(f"GPU device count: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"  Device {i}: {torch.cuda.get_device_name(i)}")

!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv


In [ ]:
# 2. Install required system packages (ffmpeg, libsndfile, espeak-ng)
!apt-get update -qq && apt-get install -y -qq ffmpeg libsndfile1 espeak-ng git > /dev/null
!ffmpeg -version | head -n 1


In [ ]:
# 3. Setup REACTOR codebase in writable /kaggle/working/REACTOR
import os, sys, shutil, subprocess
from pathlib import Path

WORK_DIR = Path("/kaggle/working/REACTOR")

# Check if repo already exists in /kaggle/working/REACTOR
if not (WORK_DIR / "src" / "reactor").is_dir():
    # Look for repo source in /kaggle/input if user attached it as a Kaggle dataset
    source_dir = None
    for candidate in Path("/kaggle/input").rglob("pyproject.toml"):
        if (candidate.parent / "src" / "reactor").is_dir():
            source_dir = candidate.parent
            break
    if source_dir is not None:
        print(f"Copying repository from read-only {source_dir} to writable {WORK_DIR}...")
        shutil.copytree(source_dir, WORK_DIR, dirs_exist_ok=True)
    else:
        print(f"Cloning latest branch from GitHub into {WORK_DIR}...")
        subprocess.run([
            "git", "clone", "-b", "feat/benchmark-fixes-and-kaggle",
            "https://github.com/0xkhush/REACTOR.git", str(WORK_DIR)
        ], check=True)

# Point process working directory and Python path to writable workspace
os.chdir(WORK_DIR)
sys.path.insert(0, str(WORK_DIR / "src"))
sys.path.insert(0, str(WORK_DIR))

assert os.access(WORK_DIR, os.W_OK), f"{WORK_DIR} must be writable!"
print(f"✅ Active repository root: {WORK_DIR} (writable: True)")


In [ ]:
# 4. Install Python dependencies
# LiveKit voice agent framework & Google plugin
!pip install -q "livekit-agents==1.3.12" "livekit-plugins-google==1.3.12"

# NVIDIA NeMo Parakeet ASR and audio utilities required by FDB-v3
!pip install -q "nemo_toolkit[asr]==2.5.3" "pydub==0.25.1" "ffmpeg-python==0.2.0"

# Install REACTOR in editable mode with development & voice extras into writable /kaggle/working/REACTOR
!pip install -q --no-build-isolation -e "/kaggle/working/REACTOR[dev,voice]"


In [ ]:
# 5. Configure credentials from Kaggle Secrets into .env.local
from pathlib import Path
import os

WORK_DIR = Path("/kaggle/working/REACTOR")
env_file = WORK_DIR / ".env.local"

livekit_url = ""
livekit_key = ""
livekit_secret = ""
google_key = ""
openai_key = ""

try:
    from kaggle_secrets import UserSecretsClient
    client = UserSecretsClient()
    livekit_url = client.get_secret("LIVEKIT_URL")
    livekit_key = client.get_secret("LIVEKIT_API_KEY")
    livekit_secret = client.get_secret("LIVEKIT_API_SECRET")
    google_key = client.get_secret("GOOGLE_API_KEY")
    try:
        openai_key = client.get_secret("OPENAI_API_KEY")
    except Exception:
        openai_key = ""
except Exception as e:
    print(f"Kaggle Secrets client notice: {e}")

# Fallback to os.environ if set manually
livekit_url = livekit_url or os.getenv("LIVEKIT_URL", "")
livekit_key = livekit_key or os.getenv("LIVEKIT_API_KEY", "")
livekit_secret = livekit_secret or os.getenv("LIVEKIT_API_SECRET", "")
google_key = google_key or os.getenv("GOOGLE_API_KEY", "")
openai_key = openai_key or os.getenv("OPENAI_API_KEY", "")

# Strip surrounding quotes and whitespace from all credentials
def _clean(val):
    return str(val).strip().strip("'\"").strip() if val else ""

livekit_url = _clean(livekit_url)
livekit_key = _clean(livekit_key)
livekit_secret = _clean(livekit_secret)
google_key = _clean(google_key)
openai_key = _clean(openai_key)

missing = []
if not livekit_url or not livekit_url.startswith("wss://"): missing.append("LIVEKIT_URL")
if not livekit_key: missing.append("LIVEKIT_API_KEY")
if not livekit_secret: missing.append("LIVEKIT_API_SECRET")
if not google_key: missing.append("GOOGLE_API_KEY")

if missing:
    raise ValueError(
        f"❌ Missing required secrets: {missing}. Please add them via Kaggle menu: Add-ons -> Secrets"
    )

# Immediate preflight verification of Google Gemini Realtime credentials
# NOTE: Jupyter already runs an event loop, so use top-level await (not asyncio.run)
import asyncio
from livekit.plugins import google
print("Verifying Google Gemini Realtime credentials...")
try:
    _llm = google.realtime.RealtimeModel(api_key=google_key)
    _sess = _llm.session()
    await asyncio.sleep(3)
    if _sess._main_atask.done() and _sess._main_atask.exception():
        raise _sess._main_atask.exception()
    await _sess.aclose()
    print("✅ Google Gemini Realtime credentials verified successfully!")
except Exception as e:
    raise RuntimeError(
        f"❌ GOOGLE_API_KEY verification failed: {e}\n"
        "Please ensure your GOOGLE_API_KEY in Kaggle Add-ons -> Secrets is a valid Gemini API key "
        "with no quotes or extra spaces."
    ) from None

env_lines = [
    "# Configured by REACTOR Kaggle Evaluation Notebook",
    "REACTOR_MODE=benchmark",
    f"LIVEKIT_URL={livekit_url}",
    f"LIVEKIT_API_KEY={livekit_key}",
    f"LIVEKIT_API_SECRET={livekit_secret}",
    f"GOOGLE_API_KEY={google_key}",
    "GOOGLE_LIVE_MODEL=gemini-2.5-flash-native-audio-preview-12-2025",
    "REACTOR_FREE_QUOTA_CONFIRMED=yes",
]
if openai_key:
    env_lines.append(f"OPENAI_API_KEY={openai_key}")

env_file.write_text("\n".join(env_lines) + "\n")
env_file.chmod(0o600)
# Export credentials to process environment for child subprocesses
os.environ["LIVEKIT_URL"] = livekit_url
os.environ["LIVEKIT_API_KEY"] = livekit_key
os.environ["LIVEKIT_API_SECRET"] = livekit_secret
os.environ["GOOGLE_API_KEY"] = google_key
os.environ["GOOGLE_LIVE_MODEL"] = "gemini-2.5-flash-native-audio-preview-12-2025"
os.environ["REACTOR_MODE"] = "benchmark"
os.environ["REACTOR_FREE_QUOTA_CONFIRMED"] = "yes"
if openai_key:
    os.environ["OPENAI_API_KEY"] = openai_key
print(f"✅ {env_file} created successfully with confirmed permissions.")


In [ ]:
# 6. Fetch upstream Full-Duplex-Bench and download released audio dataset
import subprocess, sys
from pathlib import Path

WORK_DIR = Path("/kaggle/working/REACTOR")
(WORK_DIR / "vendor").mkdir(parents=True, exist_ok=True)
cmd = [
    sys.executable, str(WORK_DIR / "scripts" / "setup_fdb.py"),
    "--destination", str(WORK_DIR / "vendor" / "Full-Duplex-Bench"),
    "--with-data"
]
print(f"Running: {' '.join(cmd)}")
subprocess.run(cmd, cwd=WORK_DIR, check=True)

# Verify dataset
dataset_path = WORK_DIR / "fdb_v3_data_released"
wav_files = sorted(dataset_path.glob("*/input.wav"))
print(f"✅ Verified {len(wav_files)}/100 benchmark input.wav recordings in {dataset_path}")
if len(wav_files) != 100:
    raise RuntimeError(f"Expected 100 input.wav recordings, found {len(wav_files)}")


In [ ]:
# 7. Run full unit test suite (203 tests) and offline preflight
import subprocess, sys
from pathlib import Path

WORK_DIR = Path("/kaggle/working/REACTOR")
print("Running pytest unit test suite...")
subprocess.run([sys.executable, "-m", "pytest", "-q"], cwd=WORK_DIR, check=True)

print("\nRunning offline benchmark preflight check...")
subprocess.run([
    sys.executable, str(WORK_DIR / "scripts" / "reproduce.py"), "--check",
    "--dataset", str(WORK_DIR / "fdb_v3_data_released"),
    "--upstream", str(WORK_DIR / "vendor" / "Full-Duplex-Bench")
], cwd=WORK_DIR, check=True)


In [ ]:
# 8. Execute Full Benchmark Reproduction
# Runs live worker, streams all 100 benchmark audio recordings,
# runs NeMo Parakeet ASR, and generates evaluation JSONs.
from pathlib import Path
import os, subprocess, sys

WORK_DIR = Path("/kaggle/working/REACTOR")
openai_key = globals().get("openai_key", "") or os.getenv("OPENAI_API_KEY", "")
cmd = [
    sys.executable, str(WORK_DIR / "scripts" / "reproduce.py"),
    "--dataset", str(WORK_DIR / "fdb_v3_data_released"),
    "--upstream", str(WORK_DIR / "vendor" / "Full-Duplex-Bench")
]

if openai_key:
    print("OPENAI_API_KEY detected: Running with --use-llm (semantic evaluator)")
    cmd.append("--use-llm")
else:
    print("Running with exact-match evaluator (no OpenAI key needed)")

print(f"🚀 Executing: {' '.join(cmd)}\n")
proc = subprocess.Popen(
    cmd,
    cwd=WORK_DIR,
    env=dict(os.environ),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

for line in iter(proc.stdout.readline, ''):
    print(line, end='', flush=True)

proc.stdout.close()
code = proc.wait()
worker_log = Path("/tmp/reactor_agent.log")
if worker_log.exists() and code != 0:
    print("\n--- LiveKit Worker Log (tail 50 lines) ---")
    lines = worker_log.read_text().splitlines()[-50:]
    print("\n".join(lines))

if code != 0:
    artifacts_exist = any((WORK_DIR / "artifacts").glob("*.json"))
    if artifacts_exist:
        print(f"\n⚠️ Notice: Reproduction script finished with return code {code}, but artifacts were generated. Continuing to summary...")
    else:
        raise RuntimeError(f"Reproduction script failed with return code {code} and produced no artifacts.")
else:
    print("\n✅ Benchmark execution and scoring finished successfully!")


In [ ]:
# 9. Summary & Metrics Display
import json
from pathlib import Path

WORK_DIR = Path("/kaggle/working/REACTOR")
artifacts_dir = WORK_DIR / "artifacts"
print("=" * 65)
print("📊 REACTOR BENCHMARK EVALUATION RESULTS")
print("=" * 65)

acc_files = sorted(artifacts_dir.glob("tool_accuracy_*.json"))
for acc_file in acc_files:
    print(f"\n--- {acc_file.name} ---")
    data = json.loads(acc_file.read_text())
    for k, v in data.items():
        if isinstance(v, (int, float, str, bool)):
            print(f"  {k:30s}: {v}")
        elif isinstance(v, dict):
            print(f"  {k}:")
            for sub_k, sub_v in v.items():
                if isinstance(sub_v, (int, float, str, bool)):
                    print(f"    {sub_k:28s}: {sub_v}")

pass_files = sorted(artifacts_dir.glob("strict_pass_rate_*.json"))
for pass_file in pass_files:
    print(f"\n--- {pass_file.name} ---")
    data = json.loads(pass_file.read_text())
    for k, v in data.items():
        if isinstance(v, (int, float, str, bool)):
            print(f"  {k:30s}: {v}")
        elif isinstance(v, dict):
            print(f"  {k}:")
            for sub_k, sub_v in v.items():
                if isinstance(sub_v, (int, float, str, bool)):
                    print(f"    {sub_k:28s}: {sub_v}")


In [ ]:
# 10. Package results into downloadable ZIP archive
import zipfile
from pathlib import Path

WORK_DIR = Path("/kaggle/working/REACTOR")
(WORK_DIR / "artifacts").mkdir(parents=True, exist_ok=True)
zip_dest = Path("/kaggle/working/REACTOR_Kaggle_Results.zip")
with zipfile.ZipFile(zip_dest, "w", zipfile.ZIP_DEFLATED) as archive:
    for file_path in (WORK_DIR / "artifacts").rglob("*"):
        if file_path.is_file():
            archive.write(file_path, arcname=f"artifacts/{file_path.relative_to(WORK_DIR / 'artifacts')}")
    for log_path in [Path("/tmp/reactor_agent.log"), Path("/tmp/agent_tool_calls.log")]:
        if log_path.is_file():
            archive.write(log_path, arcname=f"logs/{log_path.name}")

print(f"🎉 Results successfully archived: {zip_dest}")
print(f"File size: {zip_dest.stat().st_size / 1024:.2f} KB")
print("You can download this archive directly from the Kaggle Output file viewer!")
